In [ ]:
import itertools
import json
import sys
from pathlib import Path
import numpy as np
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD

ROOT_DIR = Path.cwd().parents[1]
sys.path.append(str(ROOT_DIR/"src"/"data_preprocessing"))
from normalize_fn import load

2026-01-30 20:23:19.018652: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-30 20:23:19.041820: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-30 20:23:20.112725: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-30 20:23:23.854908: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

In [ ]:
EPOCHS=1000          
BATCH_SIZE=64
ACT_FUNC=tf.nn.relu
MOMENTUM=0.5

In [ ]:
norm_options=['norm','tanh','tanh_norm']
hidden_options=[ 
    [4096,2048],[2048,1024],[4096,2048,1024],[2048,1024,512],[1024,1024]]
lr_options=[1e-2,1e-3,1e-4,1e-5]
dropout_options=[(0.0,0.0),(0.2,0.5)]
hyperparameter_grid = list(itertools.product(norm_options,hidden_options,lr_options,dropout_options))
print(f"Total no. of hyperparameter combinations: {len(hyperparameter_grid)}")

Total no. of hyperparameter combinations: 120


In [ ]:
checkpoint_file=ROOT_DIR/"hyperparam_checkpoint.json"
best_val_loss=np.inf
best_params=None
best_epoch=None
start_idx=0
if checkpoint_file.exists():
    with open(checkpoint_file,"r") as f:
        checkpoint=json.load(f)
    best_val_loss=checkpoint.get("best_val_loss",np.inf)
    best_params=checkpoint.get("best_params",None)
    best_epoch=checkpoint.get("best_epoch",None)
    start_idx=checkpoint.get("last_completed_idx",-1)+1
    print(f"Resuming from index {start_idx},best_val_loss so far:{best_val_loss}")

In [ ]:
DATA_CACHE={}
for norm in norm_options:
    X_tr,X_val, _,_,y_tr,y_val,_,_=load(norm=norm)
    DATA_CACHE[norm]=(X_tr,X_val,y_tr,y_val)
    print(f"\n{norm}")
    print("X_tr shape:",X_tr.shape)
    print("X_val shape:",X_val.shape)
    print("y_tr shape:",y_tr.shape)
    print("y_val shape:",y_val.shape)

    print("NaN in X_tr:",np.isnan(X_tr).any())
    print("Inf in X_tr:",np.isinf(X_tr).any())
    print("NaN in y_tr:",np.isnan(y_tr).any())
    print("Inf in y_tr:",np.isinf(y_tr).any())

    print("\nFirst 5 rows of X_tr:\n",X_tr[:5])
    print("First 5 elements of y_tr:\n",y_tr[:5])



norm
X_tr shape: (13884, 7063)
X_val shape: (4614, 7063)
y_tr shape: (13884, 1)
y_val shape: (4614, 1)
NaN in X_tr: False
Inf in X_tr: False
NaN in y_tr: False
Inf in y_tr: False

First 5 rows of X_tr:
 [[ 0.05809287 -1.2305294  -0.38594985 ...  0.          0.
   0.        ]
 [ 0.06163035 -1.2305294  -0.38594985 ... -0.561097   -0.68233347
   0.07810206]
 [-0.3140575  -1.2305294  -0.38594985 ...  0.4991565   1.9423046
  -0.38811162]
 [-0.15526268 -1.2305294  -0.38594985 ...  0.          0.
   0.        ]
 [-0.47901613 -1.2305294  -0.38594985 ...  0.          0.
   0.        ]]
First 5 elements of y_tr:
 [[ 7.69353  ]
 [ 7.7780533]
 [-1.1985054]
 [ 2.5956845]
 [-5.1399713]]

tanh
X_tr shape: (13884, 7063)
X_val shape: (4614, 7063)
y_tr shape: (13884, 1)
y_val shape: (4614, 1)
NaN in X_tr: False
Inf in X_tr: False
NaN in y_tr: False
Inf in y_tr: False

First 5 rows of X_tr:
 [[ 0.05802761 -0.84273285 -0.3678634  ...  0.          0.
   0.        ]
 [ 0.06155244 -0.84273285 -0.3678634  ..

In [ ]:
def moving_average(x,n):
    return np.convolve(x,np.ones(n)/n,mode='valid')

In [ ]:
for idx,(norm_type,hidden_layers,lr,(input_do,hidden_do)) in enumerate(hyperparameter_grid):
    if idx <start_idx:
        continue
    X_tr,X_val, y_tr,y_val= DATA_CACHE[norm_type]
    model =Sequential()
    for i,units in enumerate(hidden_layers):
        if i ==0:
            model.add(Dense(
                units,input_shape=(X_tr.shape[1],),activation=ACT_FUNC,kernel_initializer='he_normal'))
            if input_do >0:
                model.add(Dropout(float(input_do)))
        else:
            model.add(Dense(
                units,activation=ACT_FUNC,kernel_initializer='he_normal'))
            if hidden_do >0:
                model.add(Dropout(float(hidden_do)))
    model.add(Dense(
        1,activation='linear',kernel_initializer='he_normal'))
    model.compile(
        loss='mean_squared_error',
        optimizer=SGD(learning_rate=float(lr),momentum=MOMENTUM)
    )
    model.summary()
    history = model.fit(
        X_tr,y_tr,
        validation_data=(X_val,y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=True,
        verbose=1
    )
    train_losses= np.array(history.history['loss'])
    val_losses= np.array(history.history['val_loss'])
    local_best_epoch= int(np.argmin(val_losses))
    local_best_loss= float(val_losses[local_best_epoch])
    if local_best_loss< best_val_loss:
        best_val_loss= local_best_loss
        best_epoch =local_best_epoch + 1
        best_params ={
            "norm":norm_type,
            "hidden_layers":hidden_layers,
            "learning_rate":lr,
            "input_dropout":input_do,
            "hidden_dropout":hidden_do,
            "epochs":best_epoch
        }
    # Save checkpoint after each iteration
    checkpoint_data ={
        "last_completed_idx":idx,
        "best_val_loss":float(best_val_loss),
        "best_params":best_params,
        "best_epoch":best_epoch
    }
    with open(checkpoint_file,"w") as f:
        json.dump(checkpoint_data,f,indent=2)

    # Clear model to free memory
    del model
    K.clear_session()

I0000 00:00:1768357708.171367     183 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1768357708.409984     183 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1768357708.410015     183 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1768357708.412240     183 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1768357708.412259     183 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:0

In [ ]:
out_file = ROOT_DIR / "best_hyperparams.txt"
with open(out_file, "w") as f:
    for k, v in best_params.items():
        f.write(f"{k}: {v}\n")
    f.write(f"best_val_loss: {best_val_loss}\n")